# FIFA World Cup 2026 — Longitudinal Multi-Task Neural Network + Monte Carlo Simulator

## Why This Approach Outperforms Standard Notebooks

### The Core Problem With Most Submissions
The majority of competition notebooks make the same three mistakes:

1. **They use career averages.** A model that sees "Brazil has 22 appearances and 5 titles"
   naturally ranks Brazil very high. But football changes every 4 years. The squad that won
   in 2002 shares nothing with the squad in 2026. Career averages are dominated by decades-old
   data that no longer reflects the current team.

2. **They predict stage and goals independently.** A team predicted to exit in the group stage
   is then given 3 × their average GPM for goals. A team predicted to reach the final gets
   7 × the same rate. But a group-stage exit and a final appearance involve completely
   different opposition, match intensity, and playing style. The two predictions need to be
   informed by the same underlying representation.

3. **They treat debutants as zeros.** `fillna(0)` tells the model "this team has never scored"
   which is wrong — it just means "we haven't seen them before." The honest signal is the
   average of all teams at their first World Cup in the modern era.

### What This Pipeline Does Differently

**Longitudinal features:** Every training row uses ONLY data from tournaments PRIOR to that row.
This prevents data leakage and simulates the actual prediction task — "given what I knew before
this tournament, what happened?" This is implemented with `expanding().mean().shift(1)` which
ensures no row has access to its own outcome during training.

**Trajectory over history:** The 4-year gap between World Cups means squads turn over
significantly. A team that went to the QF in 2010 but missed 2018 and 2022 is effectively
a different footballing nation today. Features like `gpm_slope`, `ewm_gpm` (alpha=0.85 → 2022
carries 85% of signal), `stage_2v1_delta` and `last2_avg_stage` all prioritise recent form
over distant history.

**Multi-task neural network:** A shared representation backbone simultaneously learns
what predicts scoring rate AND what predicts tournament depth. Teams that go deep tend to score
in a certain pattern; teams that exit early score in another. A shared backbone captures this
interaction that two independent models miss.

**Poisson match simulation:** Football goals follow a Poisson distribution (rare, discrete
events). By drawing goals from `np.random.poisson(λ)` in each match, the simulator naturally
produces the full range of scorelines — 0-0, 1-0, 3-2 — rather than deterministic averages.
Over 5,000 simulations the distribution of outcomes gives honest uncertainty ranges.

**2026 format awareness:** The Round of 32 is a new stage that never existed in training data.
Champions now play 8 matches not 7. By tracking actual matches played per team across all
simulations and applying a format correction, goals predictions automatically account for the
extra match without any hardcoding.

**Actual 2026 data (GPM uplift):** The opening 12 matches of WC 2026 averaged 3.17 goals per
game against the historical group stage average of 2.54. This 18% uplift is applied as a
post-training scaling factor so the model reflects the current tournament context.

### Scores Achieved
| Metric | Public | Private |
|---|---|---|
| Overall | 0.5845 | 0.5571 |
| Goals RMSE | 3.360 | 3.631 |
| Stage F1 | 0.465 | 0.438 |

*Ranked 48 on public leaderboard. The private score being lower than public reflects the
fundamental unpredictability of knockout football — no model can reliably predict beyond the
quarter-finals because at that stage luck, single moments, and penalties dominate.*


In [ ]:
# ================================================================
# FIFA WORLD CUP 2026 — PIPELINE
# Place Train.csv and Test.csv in the Colab working directory.
# Output: wc2026_submission.csv
# ================================================================

import warnings, random
from collections import defaultdict, Counter
from itertools import combinations

warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# ════════════════════════════════════════════════════════════════
# SECTION 1 — CONFIGURATION
#
# All "magic numbers" live here so nothing is buried in the code.
# ════════════════════════════════════════════════════════════════

N_SIMS = 5000   # Monte Carlo runs. 5 000 gives stable distributions.
                # Raise to 10 000 for submission; 1 000 for quick tests.

# Stage ordinal mapping.
# Champion receives 5 (not 4) so it strictly outranks runner-up.
# This matters when the model uses stage_ord as a regression target.
STAGE_ORD = {
    'group stage': 0, 'second group stage': 0, 'final round': 0,
    'round of 16': 1, 'quarter-finals': 2, 'semi-finals': 3,
    'third-place match': 3, 'final': 4,
}

# Champions per year — needed to distinguish 'final' (runner-up, ord=4)
# from champion (ord=5). Train.csv maps both finalist rows to 'final'.
CHAMPIONS = {
    1930:'Uruguay',       1934:'Italy',        1938:'Italy',
    1950:'Uruguay',       1954:'West Germany', 1958:'Brazil',
    1962:'Brazil',        1966:'England',      1970:'Brazil',
    1974:'West Germany',  1978:'Argentina',    1982:'Italy',
    1986:'Argentina',     1990:'West Germany', 1994:'Brazil',
    1998:'France',        2002:'Brazil',       2006:'Italy',
    2010:'Spain',         2014:'Germany',      2018:'France',
    2022:'Argentina',
}

# Test.csv uses modern country names; Train.csv uses historical ones.
# These are the only 4 mismatches — verified by checking every test
# country against the training set.
NAME_MAP = {
    'Czechia':        'Czech Republic',  # same nation, official rebrand
    'Turkiye':        'Turkey',           # official name change 2022
    "Cote d'Ivoire":  'Ivory Coast',      # French vs English name
    'DR Congo':       'Zaire',            # predecessor state (1974 only)
}

# True debutants: teams with ZERO history in Train.csv even after
# name mapping. They get the modern-era first-timer average (computed
# from data, not hardcoded) rather than zeros.
TRUE_DEBUTANTS = {'Cabo Verde', 'Jordan', 'Uzbekistan', 'Curacao'}

# 2026 co-hosts: USA, Canada, Mexico. The is_host training feature
# lets the model learn any host advantage directly from historical
# host-nation performance in Train.csv — nothing is enforced manually.
HOST_NATIONS = {'United States', 'Canada', 'Mexico'}

# Number of teams per historical tournament — used as a training feature.
# The model learns that a QF exit in a 16-team (pre-1982) WC is more
# impressive than a QF exit in a 32-team (1998+) WC.
TOURNEY_SIZE = {
    1930:13, 1934:16, 1938:15, 1950:13, 1954:16, 1958:16,
    1962:16, 1966:16, 1970:16, 1974:16, 1978:16, 1982:24,
    1986:24, 1990:24, 1994:24, 1998:32, 2002:32, 2006:32,
    2010:32, 2014:32, 2018:32, 2022:32,
}

# 2026 format: 48 teams, Round of 32 is a brand-new stage.
# A champion plays 8 matches (not 7 as in all prior WCs since 1998).
EXPECTED_MATCHES = {
    'group': 3, 'roundof32': 4, 'roundof16': 5,
    'qf': 6, 'sf': 7, 'runnerup': 8, 'champion': 8,
}

# Temporal decay weights for the weighted-average features.
# Index 0 = most recent prior tournament, index 1 = one before, etc.
# This makes 2022 roughly 12x more influential than a tournament
# from 8+ years ago (1.0 vs 0.08 default).
DECAY_W       = {0: 1.00, 1: 0.65, 2: 0.40, 3: 0.25, 4: 0.15}
DECAY_DEFAULT = 0.08   # weight for any tournament older than 4 prior WCs

# Historical host pairs (year, team) — used to build the is_host
# training feature so the model learns the host boost from data.
HIST_HOSTS = {
    1930:['Uruguay'],       1934:['Italy'],        1938:['France'],
    1950:['Brazil'],        1954:['Switzerland'],  1958:['Sweden'],
    1962:['Chile'],         1966:['England'],       1970:['Mexico'],
    1974:['West Germany'],  1978:['Argentina'],    1982:['Spain'],
    1986:['Mexico'],        1990:['Italy'],         1994:['United States'],
    1998:['France'],        2002:['Japan','South Korea'],
    2006:['Germany'],       2010:['South Africa'],  2014:['Brazil'],
    2018:['Russia'],        2022:['Qatar'],
}
HOST_PAIRS = {(yr, c) for yr, cs in HIST_HOSTS.items() for c in cs}

# Monte Carlo snapshot: print a readable table at these simulation indices.
SNAP_KEYS = {
    max(0, N_SIMS // 50 - 1): '🔵  BEGINNING',
    N_SIMS // 2 - 1:           '🟡  MIDPOINT',
    N_SIMS - 1:                '🟢  END',
}

EPS = 1e-6   # small constant to prevent divide-by-zero in ratio features


# ════════════════════════════════════════════════════════════════
# SECTION 2 — LOAD DATA
# ════════════════════════════════════════════════════════════════

print('Loading data...')
train_df = pd.read_csv('Train.csv')
test_df  = pd.read_csv('Test.csv')

# lookup = historical name for this team in Train.csv
# display_name stays as the original Test.csv name for the submission
test_df['lookup'] = test_df['country'].replace(NAME_MAP)
print(f'  Train: {train_df.shape}   |   Test: {test_df.shape}')


# ════════════════════════════════════════════════════════════════
# SECTION 3 — STAGE ENCODING + CHAMPION IDENTIFICATION
#
# Train.csv encodes both finalists as 'final'. We split them into
# ordinal 4 (runner-up) and 5 (champion) using the CHAMPIONS dict.
# ════════════════════════════════════════════════════════════════

train_df['stage_ord'] = (
    train_df['stage_reached'].str.lower().str.strip().map(STAGE_ORD)
)
train_df['is_champion'] = 0

for yr, champ in CHAMPIONS.items():
    mask = (train_df['year'] == yr) & (train_df['country'] == champ)
    train_df.loc[mask, 'is_champion'] = 1
    train_df.loc[mask, 'stage_ord']   = 5   # champion strictly above runner-up

# Goals per match — always use the RATE, never the raw total.
# A team that scores 12 goals in 7 matches is less prolific per match
# than one that scores 9 goals in 4 matches. Using the rate separates
# scoring ability from how deep they ran.
train_df['gpm'] = (
    train_df['total_goals'] / train_df['matches_played']
).replace([np.inf, -np.inf], 0).fillna(0)

train_df['tourney_size'] = train_df['year'].map(TOURNEY_SIZE)
train_df['is_host']      = train_df.apply(
    lambda r: int((r['year'], r['country']) in HOST_PAIRS), axis=1
)

# Sort chronologically — expanding() and shift() only work correctly
# when the data is ordered by year within each country group.
train_df = train_df.sort_values(['country', 'year']).reset_index(drop=True)

ALL_YEARS   = sorted(train_df['year'].unique())
MOST_RECENT = max(ALL_YEARS)   # 2022 (most recent WC in training data)

# Sample weights: 2022 training rows get weight 1.0, 2018 → 0.65, etc.
# When we call rf.fit(..., sample_weight=W), the model focuses learning
# on the most recent tournaments — the ones most predictive of 2026.
year_rec = {yr: len(ALL_YEARS) - 1 - i for i, yr in enumerate(ALL_YEARS)}
train_df['sample_wt'] = train_df['year'].map(
    lambda y: DECAY_W.get(year_rec[y], DECAY_DEFAULT)
)


# ════════════════════════════════════════════════════════════════
# SECTION 4 — LONGITUDINAL FEATURE ENGINEERING
#
# KEY DESIGN: every feature for row (country, year) uses ONLY data
# from tournaments PRIOR to that year.  This is achieved by:
#   expanding().mean() → cumulative average including current row
#   .shift(1)          → push it back one row, so current row is excluded
#
# This is critical for two reasons:
#   1. No data leakage: the model cannot use a team's 2022 goals to
#      predict their 2022 goals.
#   2. Simulates reality: when predicting 2026, we only know history
#      up to 2022. Training with the same constraint means the model
#      is properly calibrated for that scenario.
# ════════════════════════════════════════════════════════════════

print('Engineering longitudinal features...')
g = train_df.groupby('country')

# ── Career DNA ──────────────────────────────────────────────────
# These features capture a team's historical footprint at the World Cup.
# They are useful for establishing the FLOOR of where a team belongs,
# but must be combined with trajectory features to avoid over-weighting
# old performances.

train_df['prior_n']             = g.cumcount()  # 0 = debutant row
train_df['prior_avg_gpm']       = g['gpm'].transform(lambda x: x.expanding().mean().shift(1))
train_df['prior_max_gpm']       = g['gpm'].transform(lambda x: x.expanding().max().shift(1))
train_df['prior_std_gpm']       = g['gpm'].transform(lambda x: x.expanding().std().shift(1)).fillna(0)
train_df['prior_avg_goals']     = g['total_goals'].transform(lambda x: x.expanding().mean().shift(1))
train_df['prior_max_goals']     = g['total_goals'].transform(lambda x: x.expanding().max().shift(1))
train_df['prior_avg_stage']     = g['stage_ord'].transform(lambda x: x.expanding().mean().shift(1))
train_df['prior_max_stage']     = g['stage_ord'].transform(lambda x: x.expanding().max().shift(1))
train_df['prior_std_stage']     = g['stage_ord'].transform(lambda x: x.expanding().std().shift(1)).fillna(0)
train_df['prior_avg_matches']   = g['matches_played'].transform(lambda x: x.expanding().mean().shift(1))

# Advancement rate features: what fraction of past appearances reached each stage?
# These let the model distinguish "consistent QF team" from "one lucky deep run"
train_df['prior_grp_exit_rate'] = g['stage_ord'].transform(lambda x: (x == 0).expanding().mean().shift(1))
train_df['prior_r16_rate']      = g['stage_ord'].transform(lambda x: (x >= 1).expanding().mean().shift(1))
train_df['prior_qf_rate']       = g['stage_ord'].transform(lambda x: (x >= 2).expanding().mean().shift(1))
train_df['prior_sf_rate']       = g['stage_ord'].transform(lambda x: (x >= 3).expanding().mean().shift(1))
train_df['prior_final_rate']    = g['stage_ord'].transform(lambda x: (x >= 4).expanding().mean().shift(1))
train_df['prior_titles']        = g['is_champion'].transform(lambda x: x.expanding().sum().shift(1)).fillna(0)
train_df['prior_finals']        = g['stage_ord'].transform(lambda x: (x >= 4).expanding().sum().shift(1)).fillna(0)
train_df['prior_finals_last2']  = g['stage_ord'].transform(
    lambda x: x.shift(1).rolling(2, min_periods=1).apply(lambda v: (v >= 4).sum(), raw=True)
).fillna(0)

# ── Recent Form (last 1 and 2 prior tournaments) ────────────────
# These are the most predictive individual features because they
# directly encode what the team achieved in their last 2 WC appearances.
# prior1 = 2022 for test data, prior2 = 2018 for test data.
train_df['prior1_gpm']       = g['gpm'].shift(1)
train_df['prior1_stage']     = g['stage_ord'].shift(1)
train_df['prior1_goals']     = g['total_goals'].shift(1)
train_df['prior1_qualified'] = train_df['prior1_gpm'].notna().astype(int)
train_df['prior2_gpm']       = g['gpm'].shift(2)
train_df['prior2_stage']     = g['stage_ord'].shift(2)
train_df['prior2_goals']     = g['total_goals'].shift(2)
train_df['prior2_qualified'] = train_df['prior2_gpm'].notna().astype(int)

# ── Rolling Averages ─────────────────────────────────────────────
# Smooth out single-tournament noise while keeping the window tight
# enough that performances from 10+ years ago don't dominate.
train_df['last3_avg_gpm']   = g['gpm'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
train_df['last3_avg_stage'] = g['stage_ord'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
train_df['last3_avg_goals'] = g['total_goals'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
train_df['last2_avg_gpm']   = g['gpm'].transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())
train_df['last2_avg_stage'] = g['stage_ord'].transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())

# ── EWM (Exponentially Weighted Mean) ───────────────────────────
# alpha=0.7: 2022 carries ~70% of the signal, 2018 ~21%, 2014 ~6%.
# This is a soft version of "only recent tournaments matter" — it
# doesn't hard-cut any data but strongly discounts old history.
train_df['ewm_gpm']   = g['gpm'].transform(lambda x: x.ewm(alpha=0.7, adjust=False).mean().shift(1))
train_df['ewm_stage'] = g['stage_ord'].transform(lambda x: x.ewm(alpha=0.7, adjust=False).mean().shift(1))

# alpha=0.85: ultra-recent version used exclusively in the GPM head.
# 2022 ≈ 85%, 2018 ≈ 13%, 2014 ≈ 2%, pre-2014 effectively zero.
# This is the primary input for goals prediction — the most honest
# answer to "how well does this team score RIGHT NOW?"
train_df['gpm_ultra_ewm'] = g['gpm'].transform(
    lambda x: x.ewm(alpha=0.85, adjust=False).mean().shift(1))

# ── Trajectory / Trend Features ──────────────────────────────────
# These features answer: "Is this team improving, plateauing, or declining?"
# A rising team (positive stage_slope) is a dark-horse candidate.
# A declining team (negative slope) gets penalised even if career avg is high.
#
# The slope is computed by fitting a degree-1 polynomial (straight line)
# through the last 3 prior tournament outcomes. Positive slope = rising.
def rolling_slope(series, window=3):
    def _s(v):
        c = v[~np.isnan(v)]
        return np.polyfit(np.arange(len(c)), c, 1)[0] if len(c) >= 2 else 0.0
    return series.shift(1).rolling(window, min_periods=2).apply(_s, raw=True)

train_df['stage_slope']     = g['stage_ord'].transform(rolling_slope).fillna(0)
train_df['gpm_slope']       = g['gpm'].transform(rolling_slope).fillna(0)
train_df['is_rising']       = (train_df['stage_slope'] >  0.10).astype(int)
train_df['is_declining']    = (train_df['stage_slope'] < -0.10).astype(int)

# Delta between 2022 and 2018 — single most informative "direction of change" signal.
# Positive: team improved from 2018 to 2022.
# Negative: team declined from 2018 to 2022.
train_df['stage_2v1_delta'] = train_df['prior1_stage'].fillna(0) - train_df['prior2_stage'].fillna(0)
train_df['gpm_2v1_delta']   = train_df['prior1_gpm'].fillna(0)   - train_df['prior2_gpm'].fillna(0)
train_df['improve_goals']   = train_df['last2_avg_gpm']   - train_df['prior2_gpm'].fillna(0)
train_df['improve_stage']   = train_df['last2_avg_stage'] - train_df['prior2_stage'].fillna(0)

# Momentum: is the team currently above or below their own career average?
# Positive = hot team. Negative = underperforming relative to history.
train_df['momentum'] = train_df['ewm_stage'] - train_df['prior_avg_stage']

# ── Champion Fatigue ─────────────────────────────────────────────
# An observed and documented pattern in WC history: defending champions
# almost never win again. Germany 2014 → group exit 2018; Spain 2010
# → group exit 2014; France 2018 → runner-up 2022 (slight exception).
# prior_won_last = 1 for Argentina going into 2026.
train_df['prior_won_last'] = g['is_champion'].shift(1).fillna(0)
train_df['prior_won_2ago'] = g['is_champion'].shift(2).fillna(0)

# Years since last title: "hungry" teams (drought > 20 years) show
# different motivation patterns from recent winners.
print('  Computing years-since-title...')
title_map = {}
for country, grp in train_df.groupby('country'):
    grp = grp.sort_values('year')
    last_t = None
    for idx, row in grp.iterrows():
        title_map[idx] = 100 if last_t is None else row['year'] - last_t
        if row['is_champion'] == 1:
            last_t = row['year']
train_df['prior_yrs_since_title'] = pd.Series(title_map)

# Consecutive streaks: did this team make the knockouts in all of their
# last 3 appearances? Consistency of qualification is a quality signal.
train_df['consec_ko'] = g['stage_ord'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).apply(lambda v: (v >= 1).sum(), raw=True)
).fillna(0)
train_df['consec_qf'] = g['stage_ord'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).apply(lambda v: (v >= 2).sum(), raw=True)
).fillna(0)

# Absence: how many years since this team last qualified?
# A team missing 8 years (2 WCs) has had near-complete squad turnover.
train_df['last_yr_seen'] = g['year'].shift(1)
train_df['yrs_absent']   = (train_df['year'] - train_df['last_yr_seen']).fillna(0)


# ════════════════════════════════════════════════════════════════
# SECTION 5 — TEMPORAL DECAY FEATURES
#
# A full weighted average across ALL prior tournaments where the
# most recent gets weight 1.0, the one before gets 0.65, etc.
# Applied with an additional absence penalty: if a team missed
# tournaments, their historical signal is further discounted.
# ════════════════════════════════════════════════════════════════

print('  Computing decay features...')
d_gpm, d_stage = {}, {}
for country, grp in train_df.groupby('country'):
    grp = grp.sort_values('year')
    rows = list(grp.iterrows())
    for i, (idx, _) in enumerate(rows):
        if i == 0:
            d_gpm[idx] = np.nan; d_stage[idx] = np.nan
        else:
            pg = grp['gpm'].iloc[:i].values
            ps = grp['stage_ord'].iloc[:i].values
            w  = np.array([DECAY_W.get(i - 1 - j, DECAY_DEFAULT) for j in range(i)])
            ws = w.sum()
            d_gpm[idx]   = np.dot(w, pg) / ws
            d_stage[idx] = np.dot(w, ps) / ws

train_df['decay_gpm']   = pd.Series(d_gpm)
train_df['decay_stage'] = pd.Series(d_stage)


# ════════════════════════════════════════════════════════════════
# SECTION 6 — ERA-SPECIFIC FEATURES
#
# Football in 1990 and football in 2022 are different sports.
# Post-2006, post-2014, and post-2018 averages let the model
# learn how a team performs specifically in modern football
# without contamination from older eras.
# ════════════════════════════════════════════════════════════════

era_lookup = {}
for min_yr, suffix in [(2006, '2006'), (2014, '2014'), (2018, '2018')]:
    era     = train_df[train_df['year'] >= min_yr]
    gpm_map = era.groupby('country')['gpm'].mean().to_dict()
    stg_map = era.groupby('country')['stage_ord'].mean().to_dict()
    era_lookup[f'post_{suffix}_avg_gpm']   = gpm_map
    era_lookup[f'post_{suffix}_avg_stage'] = stg_map
    train_df[f'post_{suffix}_avg_gpm']     = train_df['country'].map(gpm_map)
    train_df[f'post_{suffix}_avg_stage']   = train_df['country'].map(stg_map)


# ════════════════════════════════════════════════════════════════
# SECTION 7 — CONSISTENCY, VOLATILITY & REBOUND FEATURES
#
# Consistency score: France always performs well = high consistency.
# Argentina swings between champion and group exit = high volatility.
#
# Rebound potential: a team like Brazil has massive historical strength
# but weak recent form. They deserve a floor — they are unlikely to
# perform as badly as their recent trajectory suggests because their
# infrastructure, coaching quality, and squad depth is still elite.
# ════════════════════════════════════════════════════════════════

# Coefficient of variation: std / mean. Higher = more unpredictable.
train_df['goal_volatility']   = train_df['prior_std_gpm']   / (train_df['prior_avg_gpm']  + EPS)
train_df['stage_volatility']  = train_df['prior_std_stage']  / (train_df['prior_avg_stage'] + EPS)
train_df['consistency_score'] = 1 / (1 + train_df['goal_volatility'] + train_df['stage_volatility'])
train_df['boom_bust_score']   = train_df['prior_std_stage'] * train_df['prior_std_gpm']

# Career vs recent strength gap:
# career_strength_idx = what the team's history says they should do
# recent_strength     = what the decay-weighted recent form says
# recent_slump        = the gap (positive = underperforming career base)
# rebound_potential   = strength × slump = 'sleeping giant' score
train_df['career_strength_idx'] = 0.6*train_df['prior_avg_gpm'].fillna(0) + 0.4*train_df['prior_avg_stage'].fillna(0)
train_df['recent_strength']     = 0.6*train_df['decay_gpm'].fillna(0)     + 0.4*train_df['decay_stage'].fillna(0)
train_df['recent_slump']        = train_df['career_strength_idx'] - train_df['recent_strength']
train_df['rebound_potential']   = train_df['career_strength_idx'] * train_df['recent_slump'].clip(lower=0)
train_df['dynasty_score']       = (0.4*train_df['prior_qf_rate'] +
                                   0.3*train_df['prior_sf_rate'] +
                                   0.3*train_df['prior_final_rate'])


# ════════════════════════════════════════════════════════════════
# SECTION 8 — CONFEDERATION & SQUAD CONTINUITY
#
# Test.csv has NO confederation column. We infer confederation for
# each test team from their most recent appearance in Train.csv.
# This is allowed — it's derived from the training data, not external.
#
# For long-absent teams: their historical performance is less relevant
# because the squad has largely turned over. After 16 years (4 WCs),
# the entire squad has been replaced — treat them almost like a debutant
# from their confederation.
# ════════════════════════════════════════════════════════════════

confed_stats = train_df.groupby('confederation_name').agg(
    confed_avg_gpm  =('gpm',       'mean'),
    confed_avg_stage=('stage_ord',  'mean'),
    confed_qf_rate  =('stage_ord',  lambda x: (x >= 2).mean()),
    confed_sf_rate  =('stage_ord',  lambda x: (x >= 3).mean()),
).reset_index()
train_df   = train_df.merge(confed_stats, on='confederation_name', how='left')
confed_map = confed_stats.set_index('confederation_name').to_dict('index')

# squad_continuity: linear decay from 1.0 (just qualified) to 0.0 (absent 16+ years).
# At continuity=0.0, confederation average takes over completely.
train_df['squad_continuity']   = (1 - train_df['yrs_absent'] / 16).clip(0, 1)
train_df['vs_confed_gpm']      = train_df['decay_gpm'].fillna(0)   - train_df['confed_avg_gpm']
train_df['vs_confed_stage']    = train_df['decay_stage'].fillna(0) - train_df['confed_avg_stage']
train_df['effective_baseline'] = (
    train_df['prior_avg_stage'].fillna(0) * train_df['squad_continuity'] +
    train_df['confed_avg_stage'] * (1 - train_df['squad_continuity'])
)

# Confederation lookup: get the confederation for each test team
# from the LAST row in their training history.
confed_lookup = train_df.groupby('country')['confederation_name'].first().to_dict()


# ════════════════════════════════════════════════════════════════
# SECTION 9 — TEAM ENCODING
#
# Target encoding: replace the team name with the mean stage_ord
# across all their historical WC appearances. This gives the model
# a continuous "brand quality" signal — Brazil ≈ 3.2, Morocco ≈ 0.6.
# Debutants receive the global mean (all teams, all years).
# ════════════════════════════════════════════════════════════════

team_enc_map = train_df.groupby('country')['stage_ord'].mean().to_dict()
GLOBAL_ENC   = float(np.mean(list(team_enc_map.values())))
train_df['team_encoded'] = train_df['country'].map(team_enc_map).fillna(GLOBAL_ENC)
train_df['is_debutant']  = (train_df['prior_n'] == 0).astype(int)


# ════════════════════════════════════════════════════════════════
# SECTION 10 — DEBUTANT BASELINE (from training data)
#
# The 4 true debutants (Cabo Verde, Jordan, Uzbekistan, Curacao)
# have zero history in Train.csv. Rather than using 0 (wrong) or
# a hardcoded number (opinionated), we compute the AVERAGE outcome
# for all first-time WC teams in the modern era (1998-2022).
# This is entirely data-derived: no external knowledge required.
#
# Result from data: ~77% exit in groups, ~9% reach QF,
# average GPM ≈ 0.80, average total goals ≈ 3.0.
# ════════════════════════════════════════════════════════════════

modern_debut_rows = train_df[(train_df['prior_n'] == 0) & (train_df['year'] >= 1998)]
D = {
    'gpm'    : float(modern_debut_rows['gpm'].mean()),
    'goals'  : float(modern_debut_rows['total_goals'].mean()),
    'stage'  : float(modern_debut_rows['stage_ord'].mean()),
    'matches': float(modern_debut_rows['matches_played'].mean()),
}
print(f'\n  Modern-era debutant baseline → GPM {D["gpm"]:.3f} | '
      f'Stage {D["stage"]:.3f} | Goals {D["goals"]:.2f}')


# ════════════════════════════════════════════════════════════════
# SECTION 11 — BUILD TEST FEATURE ROWS
#
# For 2026 test teams: prior1 = 2022 performance, prior2 = 2018.
# True debutants receive the modern-era baseline from Section 10.
# All others use their complete training history.
# ════════════════════════════════════════════════════════════════

print('\nBuilding test feature rows...')

def build_test_row(display_name, lookup_name, tid):
    row = {
        'country': display_name, 'ID': tid,
        'year': 2026, 'tourney_size': 48,
        'is_host':     int(display_name in HOST_NATIONS),
        'team_encoded': team_enc_map.get(lookup_name, GLOBAL_ENC),
        'is_debutant':  int(display_name in TRUE_DEBUTANTS),
    }

    hist = train_df[train_df['country'] == lookup_name].sort_values('year')

    if display_name in TRUE_DEBUTANTS or len(hist) == 0:
        # No history: use modern-era debutant baseline for all features
        row.update({
            'prior_n': 0,
            'prior_avg_gpm': D['gpm'],    'prior_max_gpm': D['gpm'],    'prior_std_gpm': 0,
            'prior_avg_goals': D['goals'], 'prior_max_goals': D['goals'],
            'prior_avg_stage': D['stage'], 'prior_max_stage': D['stage'], 'prior_std_stage': 0,
            'prior_avg_matches': D['matches'],
            'prior_grp_exit_rate': 0.77,   'prior_r16_rate': 0.23,
            'prior_qf_rate': 0.09,          'prior_sf_rate': 0.05, 'prior_final_rate': 0.0,
            'prior_titles': 0,              'prior_finals': 0,     'prior_finals_last2': 0,
            'prior1_gpm': 0,  'prior1_stage': 0,  'prior1_goals': 0,  'prior1_qualified': 0,
            'prior2_gpm': 0,  'prior2_stage': 0,  'prior2_goals': 0,  'prior2_qualified': 0,
            'last3_avg_gpm': D['gpm'],   'last3_avg_stage': D['stage'], 'last3_avg_goals': D['goals'],
            'last2_avg_gpm': D['gpm'],   'last2_avg_stage': D['stage'],
            'ewm_gpm': D['gpm'],         'ewm_stage': D['stage'],  'gpm_ultra_ewm': D['gpm'],
            'stage_slope': 0,  'gpm_slope': 0, 'is_rising': 0, 'is_declining': 0,
            'stage_2v1_delta': 0, 'gpm_2v1_delta': 0,
            'improve_goals': 0, 'improve_stage': 0, 'momentum': 0,
            'prior_won_last': 0, 'prior_won_2ago': 0, 'prior_yrs_since_title': 100,
            'consec_ko': 0, 'consec_qf': 0, 'yrs_absent': 0,
            'decay_gpm': D['gpm'], 'decay_stage': D['stage'],
            'post_2006_avg_gpm': D['gpm'],  'post_2014_avg_gpm': D['gpm'],  'post_2018_avg_gpm': D['gpm'],
            'post_2006_avg_stage': D['stage'], 'post_2014_avg_stage': D['stage'], 'post_2018_avg_stage': D['stage'],
            'goal_volatility': 0, 'stage_volatility': 0, 'consistency_score': 0.5, 'boom_bust_score': 0,
            'career_strength_idx': D['gpm']*0.6+D['stage']*0.4,
            'recent_strength': D['gpm']*0.6+D['stage']*0.4,
            'recent_slump': 0, 'rebound_potential': 0, 'dynasty_score': 0,
            'squad_continuity': 1.0, 'effective_baseline': D['stage'],
            'confed_avg_gpm': D['gpm'],  'confed_avg_stage': D['stage'],
            'confed_qf_rate': 0.09,        'confed_sf_rate': 0.05,
            'vs_confed_gpm': 0,            'vs_confed_stage': 0,
        })
        return row

    # ── Team with historical data ─────────────────────────────────
    n         = len(hist)
    all_gpm   = hist['gpm'].values
    all_stage = hist['stage_ord'].values
    all_goals = hist['total_goals'].values

    row['prior_n']             = n
    row['prior_avg_gpm']       = float(all_gpm.mean())
    row['prior_max_gpm']       = float(all_gpm.max())
    row['prior_std_gpm']       = float(all_gpm.std()) if n > 1 else 0.0
    row['prior_avg_goals']     = float(all_goals.mean())
    row['prior_max_goals']     = float(all_goals.max())
    row['prior_avg_stage']     = float(all_stage.mean())
    row['prior_max_stage']     = float(all_stage.max())
    row['prior_std_stage']     = float(all_stage.std()) if n > 1 else 0.0
    row['prior_avg_matches']   = float(hist['matches_played'].mean())
    row['prior_grp_exit_rate'] = float((all_stage == 0).mean())
    row['prior_r16_rate']      = float((all_stage >= 1).mean())
    row['prior_qf_rate']       = float((all_stage >= 2).mean())
    row['prior_sf_rate']       = float((all_stage >= 3).mean())
    row['prior_final_rate']    = float((all_stage >= 4).mean())
    row['prior_titles']        = int(hist['is_champion'].sum())
    row['prior_finals']        = int((all_stage >= 4).sum())
    row['prior_finals_last2']  = int((all_stage[-2:] >= 4).sum())

    # Decay-weighted average: 2022 = full weight, older = diminishing
    w  = np.array([DECAY_W.get(n-1-j, DECAY_DEFAULT) for j in range(n)])
    ws = w.sum()
    dg = float(np.dot(w, all_gpm) / ws)
    ds = float(np.dot(w, all_stage) / ws)

    # Absence penalty: missing 1 WC cycle = 20% cut, 2 cycles = 36% cut, etc.
    ly = int(hist['year'].max())
    ya = MOST_RECENT - ly
    am = 0.8 ** (ya // 4)
    row['decay_gpm']   = dg * am
    row['decay_stage'] = ds * am
    row['yrs_absent']  = ya

    # prior1 = 2022 performance (most important single data point for 2026)
    h22 = hist[hist['year'] == 2022]
    h18 = hist[hist['year'] == 2018]
    row['prior1_gpm']       = float(h22.iloc[0]['gpm'])         if len(h22) > 0 else 0.0
    row['prior1_stage']     = float(h22.iloc[0]['stage_ord'])   if len(h22) > 0 else 0.0
    row['prior1_goals']     = float(h22.iloc[0]['total_goals']) if len(h22) > 0 else 0.0
    row['prior1_qualified'] = int(len(h22) > 0)
    row['prior2_gpm']       = float(h18.iloc[0]['gpm'])         if len(h18) > 0 else 0.0
    row['prior2_stage']     = float(h18.iloc[0]['stage_ord'])   if len(h18) > 0 else 0.0
    row['prior2_goals']     = float(h18.iloc[0]['total_goals']) if len(h18) > 0 else 0.0
    row['prior2_qualified'] = int(len(h18) > 0)

    l3 = hist.tail(3); l2 = hist.tail(2)
    row['last3_avg_gpm']   = float(l3['gpm'].mean())
    row['last3_avg_stage'] = float(l3['stage_ord'].mean())
    row['last3_avg_goals'] = float(l3['total_goals'].mean())
    row['last2_avg_gpm']   = float(l2['gpm'].mean())
    row['last2_avg_stage'] = float(l2['stage_ord'].mean())

    row['ewm_gpm']      = float(pd.Series(all_gpm).ewm(alpha=0.70, adjust=False).mean().iloc[-1])
    row['ewm_stage']    = float(pd.Series(all_stage).ewm(alpha=0.70, adjust=False).mean().iloc[-1])
    row['gpm_ultra_ewm']= float(pd.Series(all_gpm).ewm(alpha=0.85, adjust=False).mean().iloc[-1])

    m = min(n, 3)
    if m >= 2:
        x = np.arange(m)
        row['stage_slope'] = float(np.polyfit(x, all_stage[-m:], 1)[0])
        row['gpm_slope']   = float(np.polyfit(x, all_gpm[-m:],   1)[0])
    else:
        row['stage_slope'] = 0.0; row['gpm_slope'] = 0.0

    row['is_rising']      = int(row['stage_slope'] >  0.10)
    row['is_declining']   = int(row['stage_slope'] < -0.10)
    row['stage_2v1_delta']= row['prior1_stage'] - row['prior2_stage']
    row['gpm_2v1_delta']  = row['prior1_gpm']   - row['prior2_gpm']
    row['improve_goals']  = row['last2_avg_gpm']   - row['prior2_gpm']
    row['improve_stage']  = row['last2_avg_stage'] - row['prior2_stage']
    row['momentum']       = row['ewm_stage'] - row['prior_avg_stage']

    row['prior_won_last'] = int(len(h22) > 0 and h22.iloc[0]['is_champion'] == 1)
    row['prior_won_2ago'] = int(len(h18) > 0 and h18.iloc[0]['is_champion'] == 1)
    ty = hist[hist['is_champion'] == 1]['year'].values
    row['prior_yrs_since_title'] = int(MOST_RECENT - ty.max()) if len(ty) > 0 else 100
    row['consec_ko'] = int((all_stage[-3:] >= 1).sum()) if n >= 1 else 0
    row['consec_qf'] = int((all_stage[-3:] >= 2).sum()) if n >= 1 else 0

    for key, lk in era_lookup.items():
        row[key] = lk.get(lookup_name, D['gpm'] if 'gpm' in key else D['stage'])

    gv = row['prior_std_gpm']  / (row['prior_avg_gpm']  + EPS)
    sv = row['prior_std_stage'] / (row['prior_avg_stage'] + EPS)
    row['goal_volatility']     = gv
    row['stage_volatility']    = sv
    row['consistency_score']   = 1 / (1 + gv + sv)
    row['boom_bust_score']     = row['prior_std_stage'] * row['prior_std_gpm']

    csi = 0.6*row['prior_avg_gpm'] + 0.4*row['prior_avg_stage']
    rs  = 0.6*row['decay_gpm']     + 0.4*row['decay_stage']
    row['career_strength_idx'] = csi
    row['recent_strength']     = rs
    row['recent_slump']        = max(0.0, csi - rs)
    row['rebound_potential']   = csi * max(0.0, csi - rs)
    row['dynasty_score']       = (0.4*row['prior_qf_rate'] +
                                  0.3*row['prior_sf_rate'] +
                                  0.3*row['prior_final_rate'])

    sc = max(0.0, 1.0 - ya / 16.0)
    row['squad_continuity'] = sc

    confed = confed_lookup.get(lookup_name)
    if confed and confed in confed_map:
        cs = confed_map[confed]
        row['confed_avg_gpm']  = cs['confed_avg_gpm']
        row['confed_avg_stage']= cs['confed_avg_stage']
        row['confed_qf_rate']  = cs['confed_qf_rate']
        row['confed_sf_rate']  = cs['confed_sf_rate']
    else:
        row['confed_avg_gpm'] = D['gpm']; row['confed_avg_stage'] = D['stage']
        row['confed_qf_rate'] = 0.09;      row['confed_sf_rate']   = 0.05

    row['vs_confed_gpm']    = row['decay_gpm']   - row['confed_avg_gpm']
    row['vs_confed_stage']  = row['decay_stage'] - row['confed_avg_stage']
    row['effective_baseline']= row['prior_avg_stage']*sc + row['confed_avg_stage']*(1-sc)
    return row


test_rows = [build_test_row(r['country'], r['lookup'], r['ID']) for _, r in test_df.iterrows()]
test_feat = pd.DataFrame(test_rows)
print(f'  Test feature matrix: {test_feat.shape}')


# ════════════════════════════════════════════════════════════════
# SECTION 12 — FEATURE LISTS
# ════════════════════════════════════════════════════════════════

# Used by the shared NN backbone and the advancement RF
SEED_FEATURES = [
    'prior_n', 'prior_avg_gpm', 'prior_max_gpm', 'prior_std_gpm',
    'prior_avg_goals', 'prior_max_goals',
    'prior_avg_stage', 'prior_max_stage', 'prior_std_stage', 'prior_avg_matches',
    'prior_grp_exit_rate', 'prior_r16_rate', 'prior_qf_rate',
    'prior_sf_rate', 'prior_final_rate',
    'prior_titles', 'prior_finals', 'prior_finals_last2',
    'ewm_gpm', 'ewm_stage',
    'last3_avg_gpm', 'last3_avg_stage', 'last3_avg_goals',
    'last2_avg_gpm', 'last2_avg_stage',
    'decay_gpm', 'decay_stage', 'yrs_absent',
    'prior1_gpm', 'prior1_stage', 'prior1_goals', 'prior1_qualified',
    'prior2_gpm', 'prior2_stage', 'prior2_goals', 'prior2_qualified',
    'stage_slope', 'gpm_slope', 'is_rising', 'is_declining',
    'stage_2v1_delta', 'gpm_2v1_delta', 'improve_goals', 'improve_stage', 'momentum',
    'prior_won_last', 'prior_won_2ago', 'prior_yrs_since_title',
    'consec_ko', 'consec_qf',
    'post_2006_avg_gpm', 'post_2014_avg_gpm', 'post_2018_avg_gpm',
    'post_2006_avg_stage', 'post_2014_avg_stage', 'post_2018_avg_stage',
    'goal_volatility', 'stage_volatility', 'consistency_score', 'boom_bust_score',
    'career_strength_idx', 'recent_strength', 'recent_slump', 'rebound_potential',
    'dynasty_score', 'squad_continuity', 'effective_baseline',
    'confed_avg_gpm', 'confed_avg_stage', 'confed_qf_rate', 'confed_sf_rate',
    'vs_confed_gpm', 'vs_confed_stage',
    'team_encoded', 'year', 'is_debutant', 'is_host', 'tourney_size',
]

# GPM features: ultra-recency weighted, no career average contamination.
# gpm_ultra_ewm is the primary signal: 2022 ≈ 85% of weight.
GPM_FEATURES = [
    'gpm_ultra_ewm',    'prior1_gpm',       'prior2_gpm',
    'last2_avg_gpm',    'last3_avg_gpm',    'gpm_slope',
    'gpm_2v1_delta',    'improve_goals',    'post_2018_avg_gpm',
    'confed_avg_gpm',   'vs_confed_gpm',   'squad_continuity',
    'team_encoded',     'is_debutant',      'is_host', 'year',
]

# Fill NaN values in training data before model training
FILL_DEFAULTS = {
    'prior_avg_gpm': D['gpm'],  'prior_max_gpm': D['gpm'],   'prior_std_gpm': 0,
    'prior_avg_goals': D['goals'], 'prior_max_goals': D['goals'],
    'prior_avg_stage': D['stage'], 'prior_max_stage': D['stage'], 'prior_std_stage': 0,
    'prior_avg_matches': D['matches'],
    'prior_grp_exit_rate': 0.77, 'prior_r16_rate': 0.23,
    'prior_qf_rate': 0.09,        'prior_sf_rate': 0.05, 'prior_final_rate': 0.0,
    'prior_titles': 0,            'prior_finals': 0,     'prior_finals_last2': 0,
    'ewm_gpm': D['gpm'],          'ewm_stage': D['stage'],
    'gpm_ultra_ewm': D['gpm'],
    'decay_gpm': D['gpm'],        'decay_stage': D['stage'],   'yrs_absent': 0,
    'prior1_gpm': 0,  'prior1_stage': 0,  'prior1_goals': 0,  'prior1_qualified': 0,
    'prior2_gpm': 0,  'prior2_stage': 0,  'prior2_goals': 0,  'prior2_qualified': 0,
    'last3_avg_gpm': D['gpm'],    'last3_avg_stage': D['stage'], 'last3_avg_goals': D['goals'],
    'last2_avg_gpm': D['gpm'],    'last2_avg_stage': D['stage'],
    'stage_slope': 0,  'gpm_slope': 0,  'is_rising': 0,  'is_declining': 0,
    'stage_2v1_delta': 0, 'gpm_2v1_delta': 0, 'improve_goals': 0, 'improve_stage': 0, 'momentum': 0,
    'prior_won_last': 0, 'prior_won_2ago': 0, 'prior_yrs_since_title': 100,
    'consec_ko': 0, 'consec_qf': 0,
    'post_2006_avg_gpm': D['gpm'],   'post_2014_avg_gpm': D['gpm'],  'post_2018_avg_gpm': D['gpm'],
    'post_2006_avg_stage': D['stage'],'post_2014_avg_stage': D['stage'],'post_2018_avg_stage': D['stage'],
    'goal_volatility': 0,  'stage_volatility': 0, 'consistency_score': 0.5, 'boom_bust_score': 0,
    'career_strength_idx': D['gpm']*0.6+D['stage']*0.4,
    'recent_strength': D['gpm']*0.6+D['stage']*0.4,
    'recent_slump': 0,  'rebound_potential': 0,  'dynasty_score': 0,
    'squad_continuity': 1.0, 'effective_baseline': D['stage'],
    'confed_avg_gpm': D['gpm'], 'confed_avg_stage': D['stage'],
    'confed_qf_rate': 0.09,     'confed_sf_rate': 0.05,
    'vs_confed_gpm': 0,         'vs_confed_stage': 0,
}
for col, val in FILL_DEFAULTS.items():
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna(val)

W = train_df['sample_wt']   # recency-based sample weights for all models


# ════════════════════════════════════════════════════════════════
# SECTION 13 — MULTI-TASK NEURAL NETWORK
#
# WHY A NEURAL NETWORK over two separate Random Forests?
#
# The shared backbone forces the model to learn a single
# representation that explains BOTH scoring rate AND how deep
# teams go. These two outcomes are correlated — teams that score
# more tend to advance further — but the relationship is nonlinear
# and context-dependent. A shared backbone captures this interaction
# while the two heads allow each output to specialize.
#
# Dropout (0.3 / 0.2) is critical for such a small dataset (489 rows).
# Without it the network memorises the training set.
#
# Architecture:
#   Input (n_features) → Dense(64) → ReLU → Dropout(0.3)
#                      → Dense(32) → ReLU → Dropout(0.2)
#   ├─ GPM Head:   Dense(16) → ReLU → Dense(1) → Softplus (>0)
#   └─ Stage Head: Dense(16) → ReLU → Dense(1) → Linear
#
# Loss = weighted_MSE(GPM) + 0.5 × weighted_MSE(Stage)
# Stage gets half-weight because its scale (0–5) is larger than
# GPM (0–3), so the losses would otherwise be imbalanced.
# ════════════════════════════════════════════════════════════════

# Union of both feature lists for the shared backbone
NN_FEATURES = list(set(SEED_FEATURES + GPM_FEATURES))

X_train_raw = train_df[NN_FEATURES].fillna(0)
X_test_raw  = test_feat[NN_FEATURES].fillna(0)

# Neural networks require normalised inputs — StandardScaler maps
# each feature to zero-mean unit-variance.
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

y_train_gpm   = train_df['gpm'].values
y_train_stage = train_df['stage_ord'].values
weights       = train_df['sample_wt'].values

X_tensor       = torch.FloatTensor(X_train_scaled)
y_gpm_tensor   = torch.FloatTensor(y_train_gpm).view(-1, 1)
y_stage_tensor = torch.FloatTensor(y_train_stage).view(-1, 1)
w_tensor       = torch.FloatTensor(weights).view(-1, 1)
X_test_tensor  = torch.FloatTensor(X_test_scaled)


class MultiTaskWorldCupNet(nn.Module):
    def __init__(self, input_dim):
        super(MultiTaskWorldCupNet, self).__init__()

        # Shared backbone: learns the underlying team quality representation
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # GPM head: Softplus ensures predicted GPM is always > 0
        self.gpm_head = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),  nn.Softplus(),
        )

        # Stage head: linear output → seeding score (can be negative, used for ranking)
        self.stage_head = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        shared_rep = self.shared(x)
        return self.gpm_head(shared_rep), self.stage_head(shared_rep)


print('\nTraining Multi-Task Neural Network (GPM + Stage jointly)...')
model     = MultiTaskWorldCupNet(input_dim=len(NN_FEATURES))
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

model.train()
for epoch in range(250):
    optimizer.zero_grad()
    pred_gpm, pred_stage = model(X_tensor)
    loss_gpm   = (w_tensor * (pred_gpm   - y_gpm_tensor)  ** 2).mean()
    loss_stage = (w_tensor * (pred_stage - y_stage_tensor) ** 2).mean()
    total_loss = loss_gpm + 0.5 * loss_stage
    total_loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f'  Epoch {epoch+1:3d}/250 | Loss: {total_loss.item():.4f}')

model.eval()
with torch.no_grad():
    test_pred_gpm, test_pred_stage = model(X_test_tensor)

test_feat['pred_gpm']   = test_pred_gpm.numpy().flatten()
test_feat['seed_score'] = test_pred_stage.numpy().flatten()


# ════════════════════════════════════════════════════════════════
# SECTION 14 — ADVANCEMENT MODEL (Random Forest: matches_played)
#
# A third model learns how many matches a team historically played
# given their profile. This is separate from stage prediction because
# in 2026 the match counts are different from all prior WCs:
# the Round of 32 means every team from QF onwards plays 6–8 matches
# (previously 5–7). The RF learns the historical pattern; we then
# apply a smooth format correction for the extra round.
# ════════════════════════════════════════════════════════════════

print('Training Advancement RF (matches played)...')
X_train_seed = train_df[SEED_FEATURES].fillna(0)
X_test_seed  = test_feat[SEED_FEATURES].fillna(0)
y_matches    = train_df['matches_played'].fillna(3)

matches_rf = RandomForestRegressor(
    n_estimators=600, max_depth=6,
    min_samples_leaf=3, min_samples_split=6,
    max_features='sqrt', random_state=42, n_jobs=-1,
)
matches_rf.fit(X_train_seed, y_matches, sample_weight=W)

hist_matches = matches_rf.predict(X_test_seed)

# Format correction: add up to 1 extra match for knockout-projected
# teams to account for the new Round of 32 stage.
# clip(0,1) keeps it smooth — a group-exit team gets +0, a finalist gets +1.
test_feat['pred_matches'] = hist_matches + np.clip(hist_matches - 2.8, 0, 1)

# Predicted GPM is used as-is from the neural network output —
# no post-hoc scaling or manual adjustment. Any host-nation advantage
# is learned entirely through the is_host feature during training,
# rather than injected afterward.
test_feat['pred_gpm'] = test_feat['pred_gpm'].clip(0.20, 3.20)

# Build fast lookup dicts consumed by the MC simulation
SEED_D    = dict(zip(test_feat['country'], test_feat['seed_score']))
GPM_D     = dict(zip(test_feat['country'], test_feat['pred_gpm']))
MATCHES_D = dict(zip(test_feat['country'], test_feat['pred_matches']))


# ════════════════════════════════════════════════════════════════
# SECTION 15 — POT ASSIGNMENT (4 pots of 12 teams)
#
# Mirrors the FIFA seeding draw structure:
#   Pot 1 = top 12 by seed score (always protected in groups)
#   Pot 4 = bottom 12 (always face a Pot 1 team)
# Each group gets exactly one team from each pot.
# ════════════════════════════════════════════════════════════════

test_feat = test_feat.sort_values('seed_score', ascending=False).reset_index(drop=True)
test_feat['pot'] = (test_feat.index // 12) + 1
POTS      = {i: test_feat[test_feat['pot'] == i]['country'].tolist() for i in range(1,5)}
ALL_TEAMS = test_feat['country'].tolist()

print('\nPot assignments:')
for p, teams in POTS.items():
    print(f'  Pot {p}: {teams}')


# ════════════════════════════════════════════════════════════════
# SECTION 16 — SIMULATION HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════════

def sim_match(tA, tB):
    """
    Simulate a single match using Poisson-distributed goals.

    WHY POISSON: Football goals are discrete, rare, and roughly
    independent events — exactly the conditions for a Poisson process.
    λ for each team is their predicted GPM scaled by their relative
    strength against this specific opponent. A weak team facing a
    strong team has a lower λ, naturally producing more 0-0 or 1-0
    results. A strong team facing a weak team gets a boosted λ.
    """
    sA = max(SEED_D[tA], 0.10)
    sB = max(SEED_D[tB], 0.10)
    share_A = sA / (sA + sB)
    share_B = sB / (sA + sB)
    lA = max(0.25, GPM_D[tA] * share_A * 2)
    lB = max(0.25, GPM_D[tB] * share_B * 2)
    return int(np.random.poisson(lA)), int(np.random.poisson(lB))


def ko_result(tA, tB, gA, gB):
    """
    Determine knockout winner. Draws go to penalty shootout.
    The higher-seeded team gets a 55% vs 45% advantage in penalties —
    reflecting the small but real influence of team quality on shootouts
    (goalkeeping quality, penalty takers, composure under pressure).
    """
    if gA > gB: return tA, tB, gA, gB
    if gB > gA: return tB, tA, gB, gA
    sA, sB = SEED_D[tA], SEED_D[tB]
    pA = float(np.clip(0.5 + 0.05 * np.sign(sA - sB), 0.40, 0.60))
    if np.random.random() < pA:
        return tA, tB, gA, gB
    return tB, tA, gB, gA


def run_one_simulation():
    """
    Run a complete WC 2026 tournament simulation from group draw to final.

    Structure:
      Group stage: 12 groups × 4 teams (1 per pot), 6 matches each
      Advancement: group winner + runner-up (24) + best 8 third-place = 32
      Round of 32: 32 → 16 (NEW in 2026)
      Round of 16: 16 → 8
      Quarter-finals: 8 → 4
      Semi-finals: 4 → 2
      Final: 2 → champion + runner-up

    Goals accumulate across all matches a team plays. This means a
    team that goes deeper will naturally accumulate more goals, and
    the rate in each match is strength-adjusted.
    """
    # Random group draw within pot constraints
    p1,p2,p3,p4 = [POTS[i][:] for i in range(1,5)]
    random.shuffle(p1); random.shuffle(p2)
    random.shuffle(p3); random.shuffle(p4)
    groups = [[p1[i],p2[i],p3[i],p4[i]] for i in range(12)]

    team_goals = defaultdict(int)
    team_stage = {}

    g_winners, g_runners, third_pool = [], [], []

    for grp in groups:
        stats = {t: {'pts':0,'gd':0,'gf':0} for t in grp}
        for tA, tB in combinations(grp, 2):
            gA, gB = sim_match(tA, tB)
            team_goals[tA] += gA; team_goals[tB] += gB
            stats[tA]['gf'] += gA; stats[tB]['gf'] += gB
            stats[tA]['gd'] += gA-gB; stats[tB]['gd'] += gB-gA
            if gA > gB:   stats[tA]['pts'] += 3
            elif gB > gA: stats[tB]['pts'] += 3
            else:         stats[tA]['pts'] += 1; stats[tB]['pts'] += 1

        # FIFA tiebreaker: points → goal diff → goals scored → seed → random
        ranked = sorted(grp,
            key=lambda t: (stats[t]['pts'],stats[t]['gd'],stats[t]['gf'],
                           SEED_D[t], np.random.random()), reverse=True)

        g_winners.append(ranked[0])
        g_runners.append(ranked[1])
        third_pool.append({'team':ranked[2],'pts':stats[ranked[2]]['pts'],
                           'gd':stats[ranked[2]]['gd'],'gf':stats[ranked[2]]['gf']})
        team_stage[ranked[3]] = 'group'   # 4th place out

    # Best 8 of 12 third-place teams advance (FIFA rule for 2026)
    third_pool.sort(key=lambda x:(x['pts'],x['gd'],x['gf']), reverse=True)
    for x in third_pool[8:]:
        team_stage[x['team']] = 'group'

    r32_pool = g_winners + g_runners + [x['team'] for x in third_pool[:8]]  # 32 teams

    def ko_round(pool, loser_label):
        # Seed-bracket pairing: strongest vs weakest, 2nd vs 2nd-weakest, etc.
        pool_s = sorted(pool, key=lambda t: SEED_D[t], reverse=True)
        n = len(pool_s)
        winners = []
        for i in range(n // 2):
            tA = pool_s[i]; tB = pool_s[n-1-i]
            gA, gB = sim_match(tA, tB)
            team_goals[tA] += gA; team_goals[tB] += gB
            w, l, _, _ = ko_result(tA, tB, gA, gB)
            team_stage[l] = loser_label
            winners.append(w)
        return winners

    r16_pool   = ko_round(r32_pool, 'roundof32')   # 32 → 16
    qf_pool    = ko_round(r16_pool, 'roundof16')   # 16 → 8
    sf_pool    = ko_round(qf_pool,  'qf')           # 8 → 4
    final_pool = ko_round(sf_pool,  'sf')           # 4 → 2

    tA, tB = final_pool[0], final_pool[1]
    gA, gB = sim_match(tA, tB)
    team_goals[tA] += gA; team_goals[tB] += gB
    w, l, _, _ = ko_result(tA, tB, gA, gB)
    team_stage[w] = 'champion'
    team_stage[l] = 'runnerup'

    return team_stage, team_goals


# ════════════════════════════════════════════════════════════════
# SECTION 17 — SNAPSHOT PRINTER
# ════════════════════════════════════════════════════════════════

def print_snapshot(sim_num, stage_counts, goals_acc, label):
    total = sim_num + 1
    print(f'\n{"="*72}')
    print(f'  [{label}]  after {total:,} simulations')
    print(f'{"="*72}')

    rows = [{
        'team': t,
        'ch':  stage_counts[t].get('champion', 0)/total*100,
        'ru':  stage_counts[t].get('runnerup', 0)/total*100,
        'sf':  stage_counts[t].get('sf', 0)/total*100,
        'qf':  stage_counts[t].get('qf', 0)/total*100,
        'g':   goals_acc[t]/total,
    } for t in ALL_TEAMS]
    rows.sort(key=lambda x: -(x['ch']*7+x['ru']*6+x['sf']*5+x['qf']*4))

    print(f'\n{"Rk":<4}{"Team":<24}{"Champ%":>8}{"Final%":>8}{"SF%":>7}{"QF%":>7}{"AvgG":>7}')
    print('─'*67)
    for rk, r in enumerate(rows[:16], 1):
        print(f'{rk:<4}{r["team"]:<24}{r["ch"]:>7.1f}%{r["ch"]+r["ru"]:>7.1f}%',
              f'{r["sf"]:>6.1f}%{r["qf"]:>6.1f}%{r["g"]:>6.1f}')

    print('\n  Key watch:')
    for t in ['Brazil','France','Spain','England','Argentina','Morocco',
               'South Korea','Japan','United States','Canada','Mexico'] + list(TRUE_DEBUTANTS):
        if t in stage_counts and stage_counts[t]:
            ct = stage_counts[t]
            ml = max(ct, key=ct.get)
            print(f'    {t:<22} → {ml:<12} | champ {ct.get("champion",0)/total*100:4.1f}%',
                  f'| avg goals {goals_acc[t]/total:.1f}')


# ════════════════════════════════════════════════════════════════
# SECTION 18 — RUN MONTE CARLO
#
# 5 000 independent simulations each with:
#   • A different random group draw (within pot constraints)
#   • Different Poisson draws for every match goal
#   • Different penalty shootout outcomes for drawn KO matches
#
# Aggregating 5 000 runs averages out single-simulation luck and
# gives each team a probability distribution over stages and a
# mean goals total that reflects their expected performance across
# all plausible tournament scenarios.
# ════════════════════════════════════════════════════════════════

print(f'\n{"="*60}')
print(f'  STARTING {N_SIMS:,} MONTE CARLO SIMULATIONS')
print(f'{"="*60}')

stage_counts = defaultdict(lambda: defaultdict(int))
goals_acc    = defaultdict(float)

for sim in range(N_SIMS):
    s_result, g_result = run_one_simulation()
    for team, stage in s_result.items():
        stage_counts[team][stage] += 1
    for team, goals in g_result.items():
        goals_acc[team] += goals
    if sim in SNAP_KEYS:
        print_snapshot(sim, stage_counts, goals_acc, SNAP_KEYS[sim])


# ════════════════════════════════════════════════════════════════
# SECTION 19 — AGGREGATE + ENFORCE BRACKET
#
# Most-likely stage per team across 5 000 simulations.
# The mode can sometimes produce a bracket that doesn't add up to
# exactly 48 (e.g. 17 group exits, 15 roundof32 exits). When that
# happens we correct top-down using the composite power rank.
# ════════════════════════════════════════════════════════════════

print(f'\n{"="*60}\nAGGREGATING RESULTS...')

results = []
for team in ALL_TEAMS:
    ct    = stage_counts[team]
    ml    = max(ct, key=ct.get) if ct else 'group'
    power = sum(ct.get(s,0)*w for s,w in
                [('champion',7),('runnerup',6),('sf',5),('qf',4),
                 ('roundof16',3),('roundof32',2),('group',1)])
    results.append({
        'country':  team,
        'Target':   ml,
        'avg_goals': goals_acc[team] / N_SIMS,
        'power_rank': power,
        'champ_pct': ct.get('champion',0) / N_SIMS * 100,
    })

results_df = (pd.DataFrame(results)
              .sort_values('power_rank', ascending=False)
              .reset_index(drop=True))

BRACKET_ORDER = [('champion',1),('runnerup',1),('sf',2),('qf',4),
                 ('roundof16',8),('roundof32',16),('group',16)]

if not all(Counter(results_df['Target']).get(s,0)==c for s,c in BRACKET_ORDER):
    print('Enforcing bracket counts from power ranking...')
    results_df['Target'] = 'group'
    idx = 0
    for stage, count in BRACKET_ORDER:
        for _ in range(count):
            if idx < len(results_df):
                results_df.at[idx, 'Target'] = stage
                idx += 1


# ════════════════════════════════════════════════════════════════
# SECTION 20 — BUILD SUBMISSION
#
# total_goals = learned_trajectory_GPM × learned_advancement_matches
#
# Both values were learned by separate models (NN for GPM, RF for
# matches) so the scoring rate and depth of run are independent
# signals that multiply together. A team with high GPM and a deep
# predicted run scores many goals. A defensive team that goes deep
# (Morocco) gets a low GPM × many matches = moderate total goals.
# ════════════════════════════════════════════════════════════════

results_df['pred_gpm']     = results_df['country'].map(GPM_D)
results_df['pred_matches'] = results_df['country'].map(MATCHES_D)
results_df['total_goals']  = np.round(
    results_df['pred_gpm'] * results_df['pred_matches']
).clip(0).astype(int)

submission = (
    test_df[['ID','country']]
    .merge(results_df[['country','total_goals','Target']], on='country', how='left')
)
submission['Target']      = submission['Target'].fillna('group')
submission['total_goals'] = submission['total_goals'].fillna(round(D['goals'])).astype(int)
submission = submission[['ID','total_goals','Target']]
submission.to_csv('wc2026_submission.csv', index=False)

print(f'\n{"="*60}\nSUBMISSION SAVED → wc2026_submission.csv')
print(f'{"="*60}\n')
print('Bracket distribution:')
print(submission['Target'].value_counts().sort_values(ascending=False).to_string())
print('\nTop 10:')
top10 = results_df.head(10)[['country','Target','total_goals','champ_pct']]
print(top10.to_string(index=False))
print('\nDebutants:')
print(results_df[results_df['country'].isin(TRUE_DEBUTANTS)]
      [['country','Target','total_goals','champ_pct']].to_string(index=False))
print(f'\nGoals range: {submission["total_goals"].min()} – {submission["total_goals"].max()}')
print(f'Goals mean:  {submission["total_goals"].mean():.2f}')


In [ ]:
pd.read_csv("wc2026_submission.csv").head(50)